**STEP 1 — Install Required Libraries**

In [1]:
!pip install transformers datasets evaluate accelerate scikit-learn gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


**STEP 2 — Import Libraries**

In [2]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, f1_score
import numpy as np

**STEP 3 — Load Dataset**

In [3]:
dataset = load_dataset("ag_news")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Check dataset:

In [4]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


**STEP 4 — Understand Labels**

In [5]:
print(dataset["train"][0])

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


**STEP 5 — Load BERT Tokenizer**

Using pretrained model:

In [6]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

About BERT:

BERT

BERT is already trained on massive text corpora.
You only fine-tune it for your classification task.

**STEP 6 — Tokenize Dataset**

In [7]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

Apply tokenization:

In [8]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

**STEP 7 — Prepare Labels**

In [9]:
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

Set PyTorch format:

In [10]:
tokenized_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

**STEP 8 — Load Pretrained BERT Model**

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=4
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


**STEP 9 — Define Evaluation Metrics**

In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": accuracy,
        "f1": f1
    }

**STEP 10 — Define Training Arguments**

In [14]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


**STEP 11 — Create Trainer**

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

**STEP 12 — Train Model**

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.190927,0.181740,0.943684,0.943672
2,0.109004,0.187001,0.948421,0.948473


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15000, training_loss=0.17819939511617025, metrics={'train_runtime': 5753.4107, 'train_samples_per_second': 41.714, 'train_steps_per_second': 2.607, 'total_flos': 1.578694680576e+16, 'train_loss': 0.17819939511617025, 'epoch': 2.0})

**STEP 13 — Evaluate Model**

In [18]:
results = trainer.evaluate()

print(results)

{'eval_loss': 0.18700100481510162, 'eval_accuracy': 0.9484210526315789, 'eval_f1': 0.9484733468727846, 'eval_runtime': 63.7539, 'eval_samples_per_second': 119.208, 'eval_steps_per_second': 7.451, 'epoch': 2.0}


**STEP 14 — Save Model**

In [19]:
model.save_pretrained("saved_model")
tokenizer.save_pretrained("saved_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('saved_model/tokenizer_config.json', 'saved_model/tokenizer.json')

**STEP 15 — Test Prediction**

In [20]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="saved_model",
    tokenizer="saved_model"
)

result = classifier("Apple launches new AI-powered iPhone")

print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'LABEL_3', 'score': 0.9858286380767822}]


**STEP 16 — Verify Saved Model Folder**

In [21]:
import os

print(os.listdir("saved_model"))

['tokenizer_config.json', 'model.safetensors', 'config.json', 'tokenizer.json']


In [22]:
tokenizer.save_pretrained("saved_model")

('saved_model/tokenizer_config.json', 'saved_model/tokenizer.json')

In [23]:
import os

print(os.listdir("saved_model"))

['tokenizer_config.json', 'model.safetensors', 'config.json', 'tokenizer.json']


Test Inference:

In [26]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="saved_model",
    tokenizer="saved_model"
)

result = classifier(
    "Apple launches new AI technology"
)

print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'LABEL_3', 'score': 0.9857181310653687}]
